# 6주차 풀이 (해설)

각 문제의 접근법·핵심 개념·자주 틀리는 포인트를 정리합니다.
완성 코드는 `정답지/6주차_정답지.ipynb` 를 참조하세요.


## 1. 디코딩 파라미터로 결정론적 추출 만들기

**핵심 개념**: JSON 추출기처럼 *형식이 엄격한* 태스크는 샘플링 변동이 한 글자만 틀어져도 파싱 에러로 이어집니다. 그래서 세 축의 무작위성을 모두 꺼야 합니다.

| 파라미터 | 역할 | 결정론 설정 |
|---|---|---|
| `temperature` | 로짓을 평탄화(>1) 혹은 뾰족화(<1) | **0.0 (Greedy)** |
| `top_p` | 누적 확률 상위 p 토큰만 후보에 포함 | temperature=0 이면 사실상 무의미 → **1.0** 기본 |
| `presence_penalty` | 이미 등장한 토큰 재사용 페널티 | **0.0** (일관성) |

**자주 틀리는 포인트**
- 문제지 예시에 `top_p=0.0` 이 적혀 있지만, 대부분의 API 스펙상 top_p는 (0, 1] 범위입니다. `temperature=0` 만으로 결정론은 보장되므로 `top_p=1.0` 이 안전합니다.
- 결정론이 필요하다고 `frequency_penalty` 를 양수로 두면 오히려 반복적으로 필요한 토큰(예: 따옴표, 콜론)이 억제되어 JSON이 깨집니다.


## 2. Few-shot 예시로 출력 포맷 고정

**핵심 개념**: Zero-shot 에서 모델이 형식을 어기는 이유는 "대답의 **기대 분포**"를 못 배워서입니다. Few-shot 은 *대답 예시*를 직접 보여줘 분포를 좁힙니다.

**설계 체크리스트**
1. **시스템 프롬프트**에서 역할 + 출력 스키마 + 금지사항을 명시.
2. **Few-shot 2~5 개**: 입력(user) → 기대 출력(assistant) 쌍. 반드시 최종 포맷(JSON 배열)만 assistant 응답에 포함.
3. **다양성**: 인물 1명 / 인물 여러 명 / 기관이 불명확한 경우 등 엣지 케이스를 예시에 포함.
4. 최종 user 메시지에 실제 뉴스 텍스트를 append.

**주의**
- 예시의 assistant 응답에 "다음과 같습니다" 같은 부연이 섞이면 모델이 그 패턴을 학습합니다 → 순수 JSON만.
- 예시가 너무 유사하면 모델이 *기계적으로 복붙* 해 새 입력에 일반화하지 못합니다. 예시의 도메인을 적절히 분산.


## 3. 영수증 추출 + 재시도 로직

**핵심 개념**: 실전에서는 한 번에 완벽한 JSON이 안 오므로 *생성 → 검증 → 에러를 프롬프트에 주입해 재시도* 패턴이 표준입니다.

**시스템 프롬프트가 꼭 포함해야 할 것**
- 역할 정의 (비정형 텍스트 → JSON 추출기)
- **스키마 명세** (필드명/타입/포맷: YYYY-MM-DD, int 등)
- 출력 제약 (마크다운/코드블록/설명 금지, 순수 JSON만)

**유저 프롬프트 2가지 분기**
- **첫 시도**: 원본 텍스트 + 스키마 준수 요청.
- **재시도**: 이전 에러 메시지를 그대로 주입 → 모델이 "무엇을 고쳐야 하는지" 학습.

예: `KeyError: 'total_amount' 가 누락되었습니다` → 모델이 해당 키를 반드시 채움.

**안전장치**
- `temperature=0` 으로 재현성 확보.
- `MAX_RETRIES` 상한으로 무한 루프 방지.
- 타입 검증(`isinstance(data["total_amount"], int)`) 은 시스템 로직에서 처리 (이미 문제지에 구현되어 있음).


## 4. 감정 분류 프롬프트 e2e

**핵심 개념**: 출력이 enum(세 가지 값) + valid JSON 이어야 하므로 **Constrained Generation** 패턴.

**설계 요약**
- 시스템 프롬프트에서 ① JSON만, ② sentiment ∈ {positive, negative, neutral}, ③ reason 한 문장 을 명시.
- 스키마를 *코드처럼* 보여주면 (예: `"sentiment": "positive" | "negative" | "neutral"`) 모델이 포맷을 더 잘 맞춥니다.
- 검증 단계에서 `json.loads` 성공 여부와 `sentiment` 값이 enum 안에 있는지 함께 확인.

**실전 확장 아이디어**
- 애매한 문장(중립)에서도 모델이 억지로 positive/negative 를 고르는 경향 → 시스템 프롬프트에 "*확신이 없으면 neutral*" 규칙 추가.
- 로짓 편향을 쓸 수 있다면 `logit_bias` 로 `"neutra"` 관련 토큰을 살짝 밀어줘도 좋음.


## 5. Standard vs Chain-of-Thought

**핵심 개념**: 산술/논리가 얽힌 다단계 문제에서 Standard 프롬프트는 정답의 토큰을 *바로* 생성하려다 중간 계산을 건너뜁니다. CoT 는 모델이 추론 과정을 **외부화** 하게 해 각 단계의 실수 확률을 낮춥니다.

**문제의 정답 계산**
- 월요일: 100 - 20 + 5 = **85**
- 화요일: 85 - floor(85 × 0.1) = 85 - 8 = **77**
- 수요일: 77 + 50 - 15 = **112** ✅

**CoT 프롬프트가 잘 작동하는 장치**
1. **단계 분할 템플릿** ("월→화→수")
2. **형식 강제** ("시작 재고 → 변동 → 종료 재고")
3. **내림 명시 트리거** ("소수점 발생 시 내림")
4. **자동채점 호환 포맷** ("최종 재고: N개")

**서술형 답변은 `정답지` 마크다운 셀 참고.**


## 6. ROUGE-L / BLEU 의 본질

**핵심 개념**
- **ROUGE-L**: *정답* 기준으로 LCS(Longest Common Subsequence) 일치 → **Recall 성격**. "정답이 얼마나 복원되었는가".
- **BLEU**: *예측* 기준으로 n-gram 일치 → **Precision 성격**. "예측이 정답에서 얼마나 많이 따왔는가".

**빈칸 채우기**
```python
print(f"ROUGE-L Score: {rouge_results['rougeL_fmeasure']}")
```
`torchmetrics` 의 `ROUGEScore()` 는 dict 로 `rougeL_precision / rougeL_recall / rougeL_fmeasure` 를 모두 반환하므로 통상 F1 인 `rougeL_fmeasure` 를 사용.

**서술형 답변**
- Q4: 동의어는 n-gram 일치가 깨져 ROUGE/BLEU 가 급락.
- Q5: 해결책 = **BERTScore** (문맥 임베딩 코사인). BLEURT/BARTScore 도 계열.


## 7. ROUGE-L vs BERTScore

**핵심 개념**: 같은 두 문장에서 두 지표가 *엇갈린 점수* 를 내면 그게 바로 "의미는 같은데 표면어가 다른" 케이스. 이 실습은 그 직관을 수치로 증명하는 것.

**구현 포인트**
- `rouge_scorer.RougeScorer(['rougeL'])` 로 ROUGE-L 만 계산 후 `.fmeasure` 추출.
- `bert_score.score([pred], [ref], lang='ko')` — 한국어는 반드시 `lang='ko'` 또는 다국어 모델을 명시해야 의미론적 유사도가 제대로 잡힙니다.
- 반환값 `F1` 은 Tensor → `.item()` 으로 float 변환.

**테스트가 통과하는 이유**
- 두 문장의 어휘 중첩이 거의 없음 → ROUGE-L < 0.2 ✅.
- 의미는 사실상 동일 → BERTScore F1 > 0.8 ✅.
- 따라서 `bert_val > rouge_val` 도 성립 ✅.

**주의**
- `bert-score` 는 처음 실행 시 모델을 다운로드(수백 MB) 하므로 노트북 초기화 시간이 오래 걸릴 수 있음.
- 한국어 텍스트에 영어 BERT 를 쓰면 점수가 비정상적으로 낮게 나옵니다.


## 8. Contrastive Decoding (CD) PyTorch 구현

**핵심 아이디어**: Expert 모델과 Amateur 모델의 *로짓 차* 를 이용해, **두 모델이 공유하는 쉬운/뻔한 패턴** (Amateur 도 이미 잘 예측하는 것) 을 페널티화하고 Expert 만 아는 지식을 부각합니다.

**수식**
```
f(x) = logit_expert(x) - τ · logit_amateur(x)
valid(x) = 1 if prob_expert(x) in Top-(1-α) of vocab else 0
next_token = argmax_{x: valid(x)=1} f(x)
```

**구현 단계**
1. Expert 소프트맥스로 확률 계산.
2. **Adaptive Masking** — Expert 확률 상위권만 유효 후보. 본 정답지는 `top-k (k = round(V·(1-α)))` 로 구현. (표준 CD 는 `prob ≥ α·max_prob` 을 쓰지만, 테스트 케이스 (V=3, α=0.1) 에서 index 0 을 살리려면 top-k 방식이 직관적.)
3. 후보 내에서 `expert - τ·amateur` 계산, 후보 밖은 `-inf`.
4. softmax → argmax.

**테스트 해설**
- `expert=[12,15,4]`, `small=[10,14.5,2]`, τ=1.2 일 때
- 후보 (상위-2): index 1, 0 유지 / index 2 제거.
- `f(0) = 12 - 1.2·10 = 0`, `f(1) = 15 - 1.2·14.5 = -2.4`, `f(2) = -inf`.
- argmax = **0** — Amateur 가 index 1 을 과신하고 있기 때문에 τ 감산으로 깎임.

**외부 라이브러리 금지 주의**: `torch.topk` 와 `F.softmax` 만 사용. `transformers` 의 `top_p_filtering` 같은 편의함수 사용 금지.


## 9. DSPy Guardrail Agent

**9-1. Signature**
- `dspy.Signature` 는 "입력/출력 스키마 + 태스크 설명" 을 선언적으로 정의.
- `InputField(desc=...)`, `OutputField(desc=...)` 의 **desc 는 실제 LLM 프롬프트 생성에 쓰이므로** 구체적으로 쓸수록 성능이 올라갑니다.
- Guardrail 태스크에서는 "인젝션/탈옥/유해 요청" 같이 *무엇이 위험한지* 를 desc 에 넣어야 모델이 판단 기준을 학습합니다.

**9-2. Chain of Thought 모듈**
- `dspy.ChainOfThought(Signature)` 는 자동으로 `rationale` 필드를 추가해 **추론을 강제**합니다.
- `forward` 에서는 그대로 호출 결과를 리턴 → `result.is_safe` + `result.rationale` 모두 접근 가능.

**9-3. Generation Config**
| 키 | 값 | 이유 |
|---|---|---|
| `temperature` | 0.7 | 약간의 다양성 |
| `top_p` | 0.95 | Nucleus Sampling — 롱테일 노이즈 제거 |
| `repetition_penalty` | 1.2 | 반복 억제 (기본 1.0 보다 큼) |

OpenAI API 만 쓰는 경우엔 `repetition_penalty` 가 직접 지원되지 않으므로 `frequency_penalty=0.5~1.0` 으로 대체합니다. 로컬 LLM/HuggingFace 계열이면 `repetition_penalty` 가 표준.


## 10. Self-Correction 개선 루프

**핵심 개념**: [생성 → 비판 → 프롬프트 개정 → 재생성] 의 3단 루프. 모델이 자기 실수를 진단하고 *다음 호출의 프롬프트 자체* 를 업그레이드합니다.

**비판(critic) 프롬프트 설계 포인트**
1. 조건을 **항목별로 쪼개서** 각각 PASS/FAIL 을 강제.
2. **증거 숫자** (글자수 N, 금지어 나열, 해시태그 개수/위치) 를 같이 적게 해 추상적 평가를 방지.
3. 끝에 `[종합] PASS/FAIL — 핵심 원인` 한 줄을 강제 → 다음 단계(프롬프트 개정) 가 이 한 줄만 보고도 움직일 수 있음.

**개선(improve) 프롬프트 설계 포인트**
1. 실패한 조건을 프롬프트 **최상단** 에 "⚠️ 반드시 지킬 것" 으로 격상 → 모델의 주의(attention) 집중.
2. *측정 가능한 지시* — "작성 후 직접 글자수를 세어라" 같이 모델이 스스로 검산하도록 유도.
3. 금지 단어는 **유사어 확장** — '애플' → '애플/Apple/아이폰 제조사' 등으로 우회 방지.
4. 최종 출력은 "개선된 프롬프트 전체만" 반환하도록 강제 (메타 설명을 잘라내기 위함).

**자주 틀리는 포인트**
- critic 과 improver 의 역할 경계가 흐려서 critic 이 직접 고쳐버림 → 루프의 장점 상실.
- improver 가 원래 goal 을 버리고 자기 해석으로 rewriting → `{goal}` 을 반드시 포함시켜 보존.
- temperature 를 너무 높게 잡으면 매 시도마다 스타일이 달라져서 수렴이 느려짐 → **0.2 권장**.
